In [ ]:
questions = [
    # Machine Learning / AI (20)
    "What is overfitting?",
    "What is underfitting?",
    "Explain bias vs variance tradeoff",
    "What is gradient descent?",
    "What is stochastic gradient descent?",
    "What is regularization? (L1 vs L2)",
    "What is cross-validation?",
    "What is a confusion matrix?",
    "What is precision vs recall?",
    "What is F1 score?",
    "What is ROC-AUC?",
    "Difference between classification and regression",
    "What is a decision tree?",
    "What is a random forest?",
    "What is boosting?",
    "What is bagging?",
    "What is KNN algorithm?",
    "What is SVM?",
    "What is PCA?",
    "What is feature engineering?",

    # Statistics (10)
    "What is mean, median, and mode?",
    "What is standard deviation?",
    "What is variance?",
    "What is normal distribution?",
    "What is central limit theorem?",
    "What is hypothesis testing?",
    "What is p-value?",
    "What is confidence interval?",
    "What is correlation vs covariance?",
    "What is skewness and kurtosis?",

    # Python (10)
    "What is list vs tuple?",
    "What is dictionary in Python?",
    "What is list comprehension?",
    "What are lambda functions?",
    "What is a generator?",
    "What is difference between deep copy and shallow copy?",
    "What is exception handling?",
    "What is pandas DataFrame?",
    "What is NumPy array?",
    "What is the difference between append() and extend()?",

    # SQL (5)
    "What is SQL JOIN?",
    "What are types of joins in SQL?",
    "Difference between WHERE and HAVING",
    "What is GROUP BY?",
    "What is indexing in SQL?",

    # Case-Based / Scenario (5)
    "How would you handle missing data in a dataset?",
    "How would you detect outliers?",
    "How would you evaluate a classification model?",
    "How would you improve model performance?",
    "How would you explain a model to a non-technical person?"
]
import random

question = random.choice(questions)

print("Question:", question)

answer = input("\nYour Answer: ")

print("\nEvaluation:\n")
print(evaluate_answer(answer))

Question: What is standard deviation?

Your Answer: regularisation is method of giving penality to the dominent weight to reduce itd impacyt on the model


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Evaluation:



In [1]:
pip install faiss-cpu sentence-transformers pandas


In [2]:
import pandas as pd

# Your dataset (question | answer)
df = pd.read_csv("/content/drive/MyDrive/llama model/normalized_ds_questions.csv")

questions = df["question"].tolist()
answers = df["answer"].tolist()

In [3]:
from sentence_transformers import SentenceTransformer

# Lightweight embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

question_embeddings = embed_model.encode(questions)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import faiss
import numpy as np

dimension = question_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(question_embeddings))

print("FAISS index ready:", index.ntotal)

FAISS index ready: 1326


In [5]:
def retrieve(query, k=3):
    query_vec = embed_model.encode([query])

    distances, indices = index.search(query_vec, k)

    results = []
    for i in indices[0]:
        results.append((questions[i], answers[i]))

    return results

In [6]:
def build_prompt(query, retrieved_docs):
    context = "\n".join([f"Q: {q}\nA: {a}" for q, a in retrieved_docs])

    prompt = f"""
You are an AI interview evaluator.

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_answer(query):
    retrieved = retrieve(query)
    prompt = build_prompt(query, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)



In [8]:
query = "What is overfitting?"
response = generate_answer(query)

print(response)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are an AI interview evaluator.

Context:
Q: What is overfitting and how to avoid it?
A: Overfitting is a problem in machine learning that occurs when the model learns to fit the training data too close to the point that it starts catching up on noise and unimportant patterns. Because of this, the model performs well on training data but badly on fresh, untested data, resulting in poor generalization performance. To avoid overfitting in deep learning we can use the following techniques: Simplify the model:Overfitting may be less likely in a simpler model with fewer layers and parameters. In practical applications, it is frequently beneficial, to begin with a simple model and progressively increase its complexity until the desired performance is attained.Regularization:Regularization is a technique used in machine learning to prevent the overfitting of a model by adding a penalty term, it imposes the constraint on the weight of the model. Some of the most common regularization techn

In [2]:
def evaluate_answer(answer):
    prompt = f"""
You are a strict technical interview evaluator.

Evaluate the candidate's answer based on:
- Clarity
- Accuracy
- Depth

Answer:
{answer}

Return in this format:

Score: X/10
Clarity: Good/Average/Poor
Accuracy: Good/Average/Poor
Depth: Good/Average/Poor
Feedback:
Improvements:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,   # lower = more strict/deterministic
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [14]:
def evaluate_answer(answer,question):
    prompt = f"""
You are a strict technical interview evaluator.

Evaluate the candidate's answer.
Question:
{question}
Answer:
{answer}

Return evaluation with suggestion and correct answer also give metrics in numbers
Score: X/10
Clarity:
Accuracy::
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt part
    return full_output.replace(prompt, "").strip()

In [15]:
questions = [
    # Machine Learning / AI (20)
    "What is overfitting?",
    "What is underfitting?",
    "Explain bias vs variance tradeoff",
    "What is gradient descent?",
    "What is stochastic gradient descent?",
    "What is regularization? (L1 vs L2)",
    "What is cross-validation?",
    "What is a confusion matrix?",
    "What is precision vs recall?",
    "What is F1 score?",
    "What is ROC-AUC?",
    "Difference between classification and regression",
    "What is a decision tree?",
    "What is a random forest?",
    "What is boosting?",
    "What is bagging?",
    "What is KNN algorithm?",
    "What is SVM?",
    "What is PCA?",
    "What is feature engineering?",

    # Statistics (10)
    "What is mean, median, and mode?",
    "What is standard deviation?",
    "What is variance?",
    "What is normal distribution?",
    "What is central limit theorem?",
    "What is hypothesis testing?",
    "What is p-value?",
    "What is confidence interval?",
    "What is correlation vs covariance?",
    "What is skewness and kurtosis?",

    # Python (10)
    "What is list vs tuple?",
    "What is dictionary in Python?",
    "What is list comprehension?",
    "What are lambda functions?",
    "What is a generator?",
    "What is difference between deep copy and shallow copy?",
    "What is exception handling?",
    "What is pandas DataFrame?",
    "What is NumPy array?",
    "What is the difference between append() and extend()?",

    # SQL (5)
    "What is SQL JOIN?",
    "What are types of joins in SQL?",
    "Difference between WHERE and HAVING",
    "What is GROUP BY?",
    "What is indexing in SQL?",

    # Case-Based / Scenario (5)
    "How would you handle missing data in a dataset?",
    "How would you detect outliers?",
    "How would you evaluate a classification model?",
    "How would you improve model performance?",
    "How would you explain a model to a non-technical person?"
]
import random

question = random.choice(questions)

print("Question:", question)

answer = input("\nYour Answer: ")


print(evaluate_answer(answer,question))

Question: What is feature engineering?

Your Answer: to extract features from raw data


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Relevance:

Suggestion:
The candidate's answer is concise and to the point. However, it lacks clarity and accuracy. The candidate could have provided a more detailed explanation of what feature engineering is and how it is used in data analysis. Additionally, the candidate's answer is not relevant to the question asked.

Correct answer:
Feature engineering is the process of selecting, transforming, and creating relevant features from raw data to improve the performance of machine learning models.

Metrics:
Clarity: 5/10
Accuracy: 3/10
Relevance: 2/10

Overall score: 3/10
